# 03 · Impact of FreeSurfer versions on clinical analyses (PPMI)

Longitudinal stream, clinical cohort from `01_cohort` (`stage_two_clinical.csv`): PD-non-MCI with
UPDRS-III OFF at both visits, and HC.

* Subcortical volumes: partial correlation with UPDRS-III in PD-non-MCI (baseline volume, % change),
  ANCOVA HC vs PD-non-MCI (baseline volume, % change). Covariates: age, sex (+ time between visits).
* Vertex-wise cortical thickness: clusters from `mri_glmfit-sim` (cluster-wise p < .05), all versions.

Tables 2–3 report raw p-values; `*` = significant after Bonferroni over the 14 subcortical
regions (p < 0.05/14). p-values are never multiplied.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import fsvv

plt.rcParams.update({"figure.dpi": 80, "font.size": 8})
pd.set_option("display.max_rows", 80)

## Subcortical volumes (longitudinal stream)

In [ ]:
stage_two = pd.read_csv(fsvv.inp("stage_two_clinical.csv"))
clin, dropped = fsvv.clinical_volumes(stage_two)
print(f"{len(clin)} subjects with all 4 versions at both visits; {len(dropped)} dropped (missing longitudinal output)")
clin["dx_group"].value_counts()

In [ ]:
print("Dropped (first visit -> second visit):")
dropped.head(10)

## Statistics

In [ ]:
corr = fsvv.clinical_partial_correlations(clin)
anc = fsvv.clinical_ancova(clin)
corr.to_csv(fsvv.out("clinical_partial_corr_updrs.csv"), index=False)
anc.to_csv(fsvv.out("clinical_ancova_hc_pdnonmci.csv"), index=False)

### Check: FS5–7 only reproduces the submitted Table 2
With FS8, one PD-non-MCI participant has no longitudinal FS8 output and is dropped (124 PD-non-MCI).
Requiring only FS5–7 gives the submitted sample (125 PD-non-MCI, 106 HC) and its values
(e.g. ANCOVA, left thalamus, baseline: F = 15.696 / 15.582 / 15.627).

Note: `results_*_cov.csv` in `data/tables` (FS8 run in vinuyans/) used a different 455-row
`stage_two_clinical.csv` (147 HC, 308 PD-non-MCI), not the clinical cohort of the paper.

In [ ]:
fs567 = ["FS5", "FS6", "FS7"]
clin567, _ = fsvv.clinical_volumes(stage_two, versions=fs567)
print(len(clin567), "subjects:", clin567["dx_group"].value_counts().to_dict())
fsvv.clinical_ancova(clin567, versions=fs567, rois=["Left-Thalamus"]).query("time == 'base'").round(3)

### Table 2 — Partial correlation between UPDRS-III and subcortical volumes

In [ ]:
for time, title in [("base", "Baseline volume vs UPDRS-III"), ("long", "% change in volume vs change in UPDRS-III")]:
    print(title)
    display(fsvv.clinical_table(corr, "r", time))

### Table 3 — Group differences (ANCOVA, HC vs PD-non-MCI)

In [ ]:
for time, title in [("base", "Baseline volume"), ("long", "% change in volume")]:
    print(title)
    display(fsvv.clinical_table(anc, "F", time))

### Agreement between versions of the significant results (Sørensen–Dice)

In [ ]:
def dice_versions(results, time, alpha=fsvv.ALPHA / fsvv.N_SUBCORTICAL):
    r = results[(results["time"] == time) & results["roi"].isin(fsvv.CLINICAL_TABLE_ROIS)]
    sig = {v: set(g.loc[g["p"] < alpha, "roi"]) for v, g in r.groupby("version")}
    m = pd.DataFrame(index=fsvv.VERSIONS, columns=fsvv.VERSIONS, dtype=float)
    for a in fsvv.VERSIONS:
        for b in fsvv.VERSIONS:
            n = len(sig[a]) + len(sig[b])
            m.loc[a, b] = 2 * len(sig[a] & sig[b]) / n if n else float("nan")
    return m, {v: sorted(s) for v, s in sig.items()}


for name, res in [("ANCOVA", anc), ("partial correlation", corr)]:
    for time in ["base", "long"]:
        m, sig = dice_versions(res, time)
        print(f"{name}, {time}: significant regions per version {sig}")
        display(m.round(2))

## Vertex-wise cortical thickness (whole brain)
`group`: HC vs PD-non-MCI; `corr`: correlation with UPDRS-III in PD-non-MCI; `base`: baseline thickness;
`long`: rate of change (two-stage model). Rows = significant clusters (none = no row).

In [ ]:
clusters = fsvv.read_cluster_summaries()
clusters.to_csv(fsvv.out("vertexwise_clusters.csv"), index=False)
counts = clusters.groupby(["analysis", "time", "hemi", "version"]).size().unstack("version").reindex(
    columns=fsvv.VERSIONS).fillna(0).astype(int)
counts

In [ ]:
clusters[["version", "analysis", "time", "hemi", "ClusterNo", "Max", "Size(mm^2)", "CWP", "NVtxs", "Annot"]]